# 🏭 Manufacturing Supply Chain — Inventory Optimization
### Business Problem: Reduce stockouts, overstock costs, and improve reorder efficiency

---

**Key Questions This Notebook Answers:**
1. Which SKUs are critically understocked or overstocked?
2. What are the optimal reorder points and Economic Order Quantities (EOQ)?
3. Which suppliers are causing inventory disruptions?
4. What actionable steps will reduce holding costs and prevent stockouts?

---

## 📦 Section 1 — Setup & Synthetic Data Generation

In [27]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ── CONFIG ──────────────────────────────────────────────
N_SKUS       = 40
N_DAYS       = 365
N_SUPPLIERS  = 8
HOLDING_COST_RATE = 0.25   # 25% of unit cost per year
ORDER_COST   = 150         # Fixed cost per order ($)

CATEGORIES = ['Raw Materials', 'Components', 'Sub-Assemblies', 'Packaging', 'MRO']
SUPPLIERS   = [f'Supplier_{chr(65+i)}' for i in range(N_SUPPLIERS)]

print('✅ Libraries loaded.')
print(f'📊 Simulating {N_SKUS} SKUs × {N_DAYS} days across {N_SUPPLIERS} suppliers')

✅ Libraries loaded.
📊 Simulating 40 SKUs × 365 days across 8 suppliers


In [28]:
# ── MASTER SKU TABLE ─────────────────────────────────────
skus = pd.DataFrame({
    'SKU_ID'       : [f'SKU-{1000+i}' for i in range(N_SKUS)],
    'Category'     : np.random.choice(CATEGORIES, N_SKUS),
    'Supplier'     : np.random.choice(SUPPLIERS,  N_SKUS),
    'Unit_Cost'    : np.round(np.random.uniform(5, 500, N_SKUS), 2),
    'Lead_Time_Days': np.random.randint(3, 30, N_SKUS),
    'Safety_Stock' : np.random.randint(20, 200, N_SKUS),
    'Current_Stock': np.random.randint(0, 600, N_SKUS),
    'Max_Capacity' : np.random.randint(500, 2000, N_SKUS),
})

# Add realistic demand (some seasonal, some stable, some volatile)
demand_types = np.random.choice(['stable', 'seasonal', 'volatile'], N_SKUS, p=[0.4, 0.35, 0.25])
skus['Demand_Type'] = demand_types
base_demand = np.random.uniform(5, 80, N_SKUS)
skus['Avg_Daily_Demand'] = np.round(base_demand, 1)

print(skus.head(8).to_string(index=False))

  SKU_ID       Category   Supplier  Unit_Cost  Lead_Time_Days  Safety_Stock  Current_Stock  Max_Capacity Demand_Type  Avg_Daily_Demand
SKU-1000      Packaging Supplier_D     447.94              28           111             52          1683    seasonal              79.3
SKU-1001            MRO Supplier_A     300.96              25           148            415          1514      stable              15.5
SKU-1002 Sub-Assemblies Supplier_D     461.33              16           140            246           508      stable              43.9
SKU-1003            MRO Supplier_F      48.80               9            46            438          1756      stable              70.8
SKU-1004            MRO Supplier_B     102.01              29           140            202          1622      stable              60.6
SKU-1005     Components Supplier_B      27.39              11           135            183          1315    volatile              57.3
SKU-1006 Sub-Assemblies Supplier_A     166.04          

In [29]:
# ── DAILY TRANSACTIONS (365 days) ────────────────────────
dates = pd.date_range('2024-01-01', periods=N_DAYS, freq='D')
records = []

for _, sku in skus.iterrows():
    avg_d = sku['Avg_Daily_Demand']
    for t, date in enumerate(dates):
        # Demand model
        if sku['Demand_Type'] == 'seasonal':
            season = 1 + 0.5 * np.sin(2 * np.pi * t / 365)
            demand = max(0, np.random.poisson(avg_d * season))
        elif sku['Demand_Type'] == 'volatile':
            demand = max(0, int(np.random.normal(avg_d, avg_d * 0.6)))
        else:
            demand = max(0, np.random.poisson(avg_d))

        records.append({
            'Date'   : date,
            'SKU_ID' : sku['SKU_ID'],
            'Demand' : demand,
        })

txn = pd.DataFrame(records)

# Aggregate weekly demand for readability
txn['Week'] = txn['Date'].dt.to_period('W')
weekly = txn.groupby(['SKU_ID', 'Week'])['Demand'].sum().reset_index()
weekly['Week_Start'] = weekly['Week'].dt.start_time

print(f'✅ Generated {len(txn):,} daily transaction rows')
txn.head()

✅ Generated 14,600 daily transaction rows


,Date,SKU_ID,Demand,Week
0,2024-01-01,SKU-1000,80,2024-01-01/2024-01-07
1,2024-01-02,SKU-1000,84,2024-01-01/2024-01-07
2,2024-01-03,SKU-1000,76,2024-01-01/2024-01-07
3,2024-01-04,SKU-1000,75,2024-01-01/2024-01-07
4,2024-01-05,SKU-1000,66,2024-01-01/2024-01-07


## 📊 Section 2 — Inventory Health Overview

In [30]:
# ── COMPUTE INVENTORY KPIs ────────────────────────────────
annual_demand = txn.groupby('SKU_ID')['Demand'].sum().reset_index()
annual_demand.columns = ['SKU_ID', 'Annual_Demand']

skus = skus.merge(annual_demand, on='SKU_ID')

# Days of Supply remaining
skus['Days_of_Supply'] = np.where(
    skus['Avg_Daily_Demand'] > 0,
    skus['Current_Stock'] / skus['Avg_Daily_Demand'],
    999
)

# Inventory Status
def classify_inventory(row):
    dos = row['Days_of_Supply']
    lt  = row['Lead_Time_Days']
    if row['Current_Stock'] == 0:
        return 'STOCKOUT'
    elif dos < lt:
        return 'CRITICAL'
    elif dos < lt * 1.5:
        return 'LOW'
    elif row['Current_Stock'] > row['Max_Capacity'] * 0.85:
        return 'OVERSTOCK'
    else:
        return 'HEALTHY'

skus['Inventory_Status'] = skus.apply(classify_inventory, axis=1)

status_counts = skus['Inventory_Status'].value_counts().reset_index()
print('\n📋 Inventory Status Summary:')
print(status_counts.to_string(index=False))


📋 Inventory Status Summary:
Inventory_Status  count
        CRITICAL     32
         HEALTHY      5
             LOW      3


In [31]:
# ── CHART 1: Inventory Status Distribution ───────────────
color_map = {
    'STOCKOUT' : '#EF4444',
    'CRITICAL' : '#F97316',
    'LOW'      : '#EAB308',
    'HEALTHY'  : '#22C55E',
    'OVERSTOCK': '#3B82F6',
}

fig = px.pie(
    status_counts,
    names='Inventory_Status',
    values='count',
    title='📦 Inventory Health Status — All SKUs',
    color='Inventory_Status',
    color_discrete_map=color_map,
    hole=0.45
)
fig.update_layout(template='plotly_dark', height=420)
fig.show()

In [32]:
# ── CHART 2: Stock Level vs Safety Stock (Scatter) ────────
fig = px.scatter(
    skus,
    x='Current_Stock',
    y='Days_of_Supply',
    color='Inventory_Status',
    size='Unit_Cost',
    hover_data=['SKU_ID', 'Category', 'Supplier', 'Lead_Time_Days'],
    color_discrete_map=color_map,
    title='🔍 Current Stock vs Days of Supply (bubble = unit cost)',
    labels={'Current_Stock': 'Current Stock (units)', 'Days_of_Supply': 'Days of Supply Remaining'}
)
fig.add_hline(y=14, line_dash='dash', line_color='orange', annotation_text='⚠️ 2-Week Warning Line')
fig.add_hline(y=7,  line_dash='dash', line_color='red',    annotation_text='🚨 1-Week Critical Line')
fig.update_layout(template='plotly_dark', height=480)
fig.show()

## 📐 Section 3 — Economic Order Quantity (EOQ) Analysis

In [33]:
# ── EOQ FORMULA: sqrt(2 * D * S / H) ─────────────────────
# D = Annual demand, S = Order cost, H = Holding cost per unit per year

skus['Holding_Cost_Per_Unit'] = skus['Unit_Cost'] * HOLDING_COST_RATE

skus['EOQ'] = np.sqrt(
    (2 * skus['Annual_Demand'] * ORDER_COST) / skus['Holding_Cost_Per_Unit']
).round(0).astype(int)

# Reorder Point = (Avg Daily Demand × Lead Time) + Safety Stock
skus['Reorder_Point'] = (
    skus['Avg_Daily_Demand'] * skus['Lead_Time_Days'] + skus['Safety_Stock']
).round(0).astype(int)

# Reorder NOW flag
skus['Needs_Reorder'] = skus['Current_Stock'] <= skus['Reorder_Point']

# Annual Holding Cost at current stock
skus['Annual_Holding_Cost'] = (skus['Current_Stock'] * skus['Holding_Cost_Per_Unit']).round(2)

print(f"\n🚨 SKUs requiring IMMEDIATE reorder: {skus['Needs_Reorder'].sum()}")
print(f"💰 Total annual holding cost (current): ${skus['Annual_Holding_Cost'].sum():,.0f}")

skus[['SKU_ID','Category','EOQ','Reorder_Point','Current_Stock','Needs_Reorder']].head(10)


🚨 SKUs requiring IMMEDIATE reorder: 35
💰 Total annual holding cost (current): $677,239


,SKU_ID,Category,EOQ,Reorder_Point,Current_Stock,Needs_Reorder
0,SKU-1000,Packaging,280,2331,52,True
1,SKU-1001,MRO,150,536,415,True
2,SKU-1002,Sub-Assemblies,204,842,246,True
3,SKU-1003,MRO,795,683,438,True
4,SKU-1004,MRO,509,1897,202,True
5,SKU-1005,Components,961,765,183,True
6,SKU-1006,Sub-Assemblies,389,1003,122,True
7,SKU-1007,Sub-Assemblies,262,666,400,True
8,SKU-1008,Sub-Assemblies,289,930,293,True
9,SKU-1009,MRO,264,944,279,True


In [34]:
# ── CHART 3: EOQ vs Current Stock ────────────────────────
fig = go.Figure()

fig.add_trace(go.Bar(
    name='Current Stock',
    x=skus['SKU_ID'],
    y=skus['Current_Stock'],
    marker_color='#60A5FA',
    opacity=0.8
))
fig.add_trace(go.Bar(
    name='EOQ (Optimal Order Qty)',
    x=skus['SKU_ID'],
    y=skus['EOQ'],
    marker_color='#34D399',
    opacity=0.8
))
fig.add_trace(go.Scatter(
    name='Reorder Point',
    x=skus['SKU_ID'],
    y=skus['Reorder_Point'],
    mode='markers',
    marker=dict(color='#F59E0B', size=7, symbol='diamond'),
))

fig.update_layout(
    title='📐 EOQ vs Current Stock vs Reorder Point per SKU',
    barmode='group',
    template='plotly_dark',
    height=500,
    xaxis_tickangle=-45,
    xaxis_title='SKU',
    yaxis_title='Units'
)
fig.show()

## 📈 Section 4 — Demand Forecasting & Trend Analysis

In [35]:
# ── TOP 6 HIGH-VALUE SKUs by demand × unit cost ───────────
skus['Value_Score'] = skus['Annual_Demand'] * skus['Unit_Cost']
top6 = skus.nlargest(6, 'Value_Score')['SKU_ID'].tolist()

top6_weekly = weekly[weekly['SKU_ID'].isin(top6)]

fig = px.line(
    top6_weekly,
    x='Week_Start',
    y='Demand',
    color='SKU_ID',
    title='📈 Weekly Demand Trends — Top 6 High-Value SKUs',
    labels={'Demand': 'Weekly Demand (units)', 'Week_Start': 'Week'},
)
fig.update_layout(template='plotly_dark', height=440)
fig.show()

In [36]:
# ── CHART 5: Demand Variability (CV) ─────────────────────
demand_stats = txn.groupby('SKU_ID')['Demand'].agg(['mean', 'std']).reset_index()
demand_stats.columns = ['SKU_ID', 'Mean_Demand', 'Std_Demand']
demand_stats['CV'] = (demand_stats['Std_Demand'] / demand_stats['Mean_Demand']).round(3)

skus = skus.merge(demand_stats[['SKU_ID', 'CV']], on='SKU_ID')

fig = px.scatter(
    skus,
    x='Avg_Daily_Demand',
    y='CV',
    color='Category',
    size='Annual_Holding_Cost',
    hover_data=['SKU_ID', 'Supplier', 'Inventory_Status'],
    title='📉 Demand Variability (CV) vs Avg Daily Demand',
    labels={'CV': 'Coefficient of Variation (volatility)', 'Avg_Daily_Demand': 'Avg Daily Demand'}
)
fig.add_hline(y=0.5, line_dash='dot', line_color='orange', annotation_text='High Volatility Threshold')
fig.update_layout(template='plotly_dark', height=460)
fig.show()

## 🏭 Section 5 — Supplier Performance Analysis

In [37]:
# ── SUPPLIER METRICS ──────────────────────────────────────
supplier_df = skus.groupby('Supplier').agg(
    SKU_Count        = ('SKU_ID', 'count'),
    Avg_Lead_Time    = ('Lead_Time_Days', 'mean'),
    Stockout_SKUs    = ('Inventory_Status', lambda x: (x == 'STOCKOUT').sum()),
    Critical_SKUs    = ('Inventory_Status', lambda x: (x == 'CRITICAL').sum()),
    Total_Hold_Cost  = ('Annual_Holding_Cost', 'sum'),
    Avg_Unit_Cost    = ('Unit_Cost', 'mean'),
).reset_index()

supplier_df['Risk_Score'] = (
    supplier_df['Stockout_SKUs'] * 3 +
    supplier_df['Critical_SKUs'] * 2 +
    supplier_df['Avg_Lead_Time'] * 0.5
).round(1)

print(supplier_df.sort_values('Risk_Score', ascending=False).to_string(index=False))

  Supplier  SKU_Count  Avg_Lead_Time  Stockout_SKUs  Critical_SKUs  Total_Hold_Cost  Avg_Unit_Cost  Risk_Score
Supplier_B         11      19.000000              0              9        139373.47     240.927273        27.5
Supplier_D         10      18.000000              0              9        120102.94     212.302000        27.0
Supplier_F          6      14.833333              0              5        126754.54     256.330000        17.4
Supplier_G          4      13.500000              0              4         76903.08     274.917500        14.8
Supplier_E          3      14.333333              0              2         81884.18     332.996667        11.2
Supplier_A          3      18.333333              0              1         81537.30     273.963333        11.2
Supplier_H          2      17.000000              0              1         49906.64     214.455000        10.5
Supplier_C          1      15.000000              0              1           776.86       7.730000         9.5


In [38]:
# ── CHART 6: Supplier Risk vs Lead Time ───────────────────
fig = px.scatter(
    supplier_df,
    x='Avg_Lead_Time',
    y='Risk_Score',
    size='Total_Hold_Cost',
    color='Risk_Score',
    text='Supplier',
    title='🏭 Supplier Risk Score vs Avg Lead Time (bubble = holding cost)',
    color_continuous_scale='RdYlGn_r',
    labels={'Avg_Lead_Time': 'Avg Lead Time (days)', 'Risk_Score': 'Risk Score'}
)
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_dark', height=460)
fig.show()

In [39]:
# ── CHART 7: Holding Cost by Category + Supplier ─────────
cat_sup = skus.groupby(['Category', 'Supplier'])['Annual_Holding_Cost'].sum().reset_index()

fig = px.sunburst(
    cat_sup,
    path=['Category', 'Supplier'],
    values='Annual_Holding_Cost',
    title='💰 Annual Holding Cost Breakdown by Category & Supplier',
    color='Annual_Holding_Cost',
    color_continuous_scale='Blues'
)
fig.update_layout(template='plotly_dark', height=520)
fig.show()

## 🔴 Section 6 — ABC Analysis (Inventory Classification)

In [40]:
# ── ABC ANALYSIS ─────────────────────────────────────────
# A = top 70% of value, B = next 20%, C = bottom 10%
skus_abc = skus.sort_values('Value_Score', ascending=False).copy()
skus_abc['Cumulative_Value'] = skus_abc['Value_Score'].cumsum()
total_val = skus_abc['Value_Score'].sum()
skus_abc['Cumulative_Pct'] = skus_abc['Cumulative_Value'] / total_val * 100

def abc_class(pct):
    if pct <= 70: return 'A — High Value'
    elif pct <= 90: return 'B — Medium Value'
    else: return 'C — Low Value'

skus_abc['ABC_Class'] = skus_abc['Cumulative_Pct'].apply(abc_class)

abc_summary = skus_abc.groupby('ABC_Class').agg(
    SKU_Count      = ('SKU_ID', 'count'),
    Total_Value    = ('Value_Score', 'sum'),
    Avg_Hold_Cost  = ('Annual_Holding_Cost', 'mean'),
).reset_index()

print(abc_summary.to_string(index=False))

       ABC_Class  SKU_Count  Total_Value  Avg_Hold_Cost
  A — High Value         14 103157933.03   26683.752143
B — Medium Value         10  28495579.39   13926.980000
   C — Low Value         16  16391083.51   10274.792500


In [41]:
# ── CHART 8: ABC Pareto Curve ─────────────────────────────
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(go.Bar(
    x=skus_abc['SKU_ID'],
    y=skus_abc['Value_Score'],
    name='Annual Value ($)',
    marker_color=skus_abc['ABC_Class'].map({
        'A — High Value': '#EF4444',
        'B — Medium Value': '#F97316',
        'C — Low Value': '#22C55E'
    }),
), secondary_y=False)

fig.add_trace(go.Scatter(
    x=skus_abc['SKU_ID'],
    y=skus_abc['Cumulative_Pct'],
    name='Cumulative %',
    line=dict(color='white', width=2),
    mode='lines'
), secondary_y=True)

fig.add_hline(y=70,  secondary_y=True, line_dash='dash', line_color='red',    annotation_text='A/B cut-off (70%)')
fig.add_hline(y=90,  secondary_y=True, line_dash='dash', line_color='orange', annotation_text='B/C cut-off (90%)')

fig.update_layout(
    title='📊 ABC Pareto Analysis — SKU Annual Value',
    template='plotly_dark',
    height=500,
    xaxis_tickangle=-45,
    showlegend=True
)
fig.update_yaxes(title_text='Annual Value ($)', secondary_y=False)
fig.update_yaxes(title_text='Cumulative %', secondary_y=True)
fig.show()

## 🎯 Section 7 — Actionable Recommendations Dashboard

In [42]:
# ── BUILD RECOMMENDATION TABLE ────────────────────────────
skus_full = skus_abc.copy()

def generate_action(row):
    actions = []
    status = row['Inventory_Status']
    abc    = row['ABC_Class']
    cv     = row['CV']

    if status == 'STOCKOUT':
        actions.append(f'🚨 EMERGENCY ORDER: Place order for {row["EOQ"]} units immediately')
    elif status == 'CRITICAL':
        actions.append(f'⚠️ URGENT REORDER: Stock below lead-time buffer — order {row["EOQ"]} units')
    elif status == 'LOW':
        actions.append(f'📦 REORDER SOON: Approaching reorder point of {row["Reorder_Point"]} units')
    elif status == 'OVERSTOCK':
        excess = int(row['Current_Stock'] - row['Max_Capacity'] * 0.7)
        cost   = round(excess * row['Holding_Cost_Per_Unit'], 0)
        actions.append(f'💰 REDUCE STOCK: {excess} units excess, costing ${cost}/yr in holding')

    if cv > 0.5 and abc == 'A — High Value':
        actions.append('📈 HIGH VOLATILITY: Increase safety stock or switch to vendor-managed inventory')

    if row['Lead_Time_Days'] > 20 and status in ['CRITICAL', 'LOW']:
        actions.append(f'🏭 LONG LEAD TIME ({row["Lead_Time_Days"]}d): Consider dual-sourcing from a backup supplier')

    return ' | '.join(actions) if actions else '✅ No action required'

skus_full['Recommended_Action'] = skus_full.apply(generate_action, axis=1)

priority_df = skus_full[skus_full['Recommended_Action'] != '✅ No action required'].sort_values(
    ['Inventory_Status', 'Value_Score'], ascending=[True, False]
)[['SKU_ID', 'Category', 'Supplier', 'ABC_Class', 'Inventory_Status',
   'Current_Stock', 'Reorder_Point', 'EOQ', 'Days_of_Supply', 'Recommended_Action']]

print(f'🎯 Total SKUs requiring action: {len(priority_df)}')
priority_df.head(12)

🎯 Total SKUs requiring action: 35


,SKU_ID,Category,Supplier,ABC_Class,Inventory_Status,Current_Stock,Reorder_Point,EOQ,Days_of_Supply,Recommended_Action
0,SKU-1000,Packaging,Supplier_D,A — High Value,CRITICAL,52,2331,280,0.655738,⚠️ URGENT REORDER: Stock below lead-time buffe...
35,SKU-1035,MRO,Supplier_B,A — High Value,CRITICAL,1,984,247,0.015949,⚠️ URGENT REORDER: Stock below lead-time buffe...
9,SKU-1009,MRO,Supplier_B,A — High Value,CRITICAL,279,944,264,4.246575,⚠️ URGENT REORDER: Stock below lead-time buffe...
16,SKU-1016,Packaging,Supplier_E,A — High Value,CRITICAL,325,513,217,6.052142,⚠️ URGENT REORDER: Stock below lead-time buffe...
17,SKU-1017,MRO,Supplier_H,A — High Value,CRITICAL,463,1619,254,8.038194,⚠️ URGENT REORDER: Stock below lead-time buffe...
2,SKU-1002,Sub-Assemblies,Supplier_D,A — High Value,CRITICAL,246,842,204,5.603645,⚠️ URGENT REORDER: Stock below lead-time buffe...
12,SKU-1012,MRO,Supplier_G,A — High Value,CRITICAL,143,732,342,1.945578,⚠️ URGENT REORDER: Stock below lead-time buffe...
23,SKU-1023,Raw Materials,Supplier_B,A — High Value,CRITICAL,147,757,236,3.037190,⚠️ URGENT REORDER: Stock below lead-time buffe...
14,SKU-1014,Packaging,Supplier_G,A — High Value,CRITICAL,123,1108,215,2.887324,⚠️ URGENT REORDER: Stock below lead-time buffe...
28,SKU-1028,Packaging,Supplier_F,A — High Value,CRITICAL,150,317,265,3.042596,⚠️ URGENT REORDER: Stock below lead-time buffe...


In [43]:
# ── CHART 9: Priority Heatmap (SKU × Status × Value) ─────
heat_data = skus_full.pivot_table(
    index='Category',
    columns='Inventory_Status',
    values='SKU_ID',
    aggfunc='count',
    fill_value=0
)

fig = px.imshow(
    heat_data,
    text_auto=True,
    color_continuous_scale='RdYlGn_r',
    title='🔥 Inventory Status Heatmap — Category × Status (SKU Count)',
    aspect='auto'
)
fig.update_layout(template='plotly_dark', height=420)
fig.show()

In [44]:
# ── CHART 10: Cost Savings Potential ─────────────────────
overstock_skus = skus_full[skus_full['Inventory_Status'] == 'OVERSTOCK'].copy()
overstock_skus['Excess_Units'] = (
    overstock_skus['Current_Stock'] - overstock_skus['Max_Capacity'] * 0.7
).clip(lower=0).astype(int)
overstock_skus['Savings_If_Reduced'] = (
    overstock_skus['Excess_Units'] * overstock_skus['Holding_Cost_Per_Unit']
).round(2)

if len(overstock_skus) > 0:
    fig = px.bar(
        overstock_skus.sort_values('Savings_If_Reduced', ascending=True),
        x='Savings_If_Reduced',
        y='SKU_ID',
        orientation='h',
        color='Category',
        title=f'💰 Potential Annual Savings from Reducing Overstock (Total: ${overstock_skus["Savings_If_Reduced"].sum():,.0f})',
        labels={'Savings_If_Reduced': 'Annual Saving ($)', 'SKU_ID': 'SKU'}
    )
    fig.update_layout(template='plotly_dark', height=420)
    fig.show()
else:
    print('No overstock SKUs detected in this simulation run.')

No overstock SKUs detected in this simulation run.


## ✅ Section 8 — Executive Summary & Action Plan

In [45]:
# ── EXECUTIVE SUMMARY ────────────────────────────────────
stockouts    = (skus_full['Inventory_Status'] == 'STOCKOUT').sum()
critical     = (skus_full['Inventory_Status'] == 'CRITICAL').sum()
overstock    = (skus_full['Inventory_Status'] == 'OVERSTOCK').sum()
total_hold   = skus_full['Annual_Holding_Cost'].sum()
overstock_savings = overstock_skus['Savings_If_Reduced'].sum() if len(overstock_skus) > 0 else 0
high_risk_sup = supplier_df.sort_values('Risk_Score', ascending=False).iloc[0]['Supplier']
top_reorder_skus = priority_df[priority_df['Inventory_Status'].isin(['STOCKOUT','CRITICAL'])]['SKU_ID'].tolist()

summary = f"""
╔══════════════════════════════════════════════════════════════════╗
║         🏭 MANUFACTURING INVENTORY OPTIMIZATION — SUMMARY        ║
╠══════════════════════════════════════════════════════════════════╣
║  📊 CURRENT STATE                                                ║
║  • Total SKUs Analyzed   : {N_SKUS}                                    ║
║  • Stockout SKUs         : {stockouts} (IMMEDIATE action required)          ║
║  • Critical SKUs         : {critical} (order within lead time)              ║
║  • Overstock SKUs        : {overstock}                                     ║
║  • Total Annual Hold Cost: ${total_hold:>10,.0f}                       ║
╠══════════════════════════════════════════════════════════════════╣
║  🎯 TOP ACTIONS                                                  ║
║  1. 🚨 Emergency Orders: {len(top_reorder_skus)} SKUs need immediate replenishment   ║
║  2. 💰 Overstock Reduction: Save ~${overstock_savings:,.0f}/yr                ║
║  3. 🏭 Audit Supplier: {high_risk_sup} has highest risk score                ║
║  4. 📐 Adopt EOQ model for all A-class SKUs                      ║
║  5. 📈 Add safety stock buffer for high-CV (volatile) SKUs       ║
║  6. 🔄 Set automated reorder alerts at computed Reorder Points   ║
╠══════════════════════════════════════════════════════════════════╣
║  📋 STRATEGIC RECOMMENDATIONS                                    ║
║  • Implement ABC tiered review: A weekly, B bi-weekly, C monthly ║
║  • Dual-source suppliers with lead time > 20 days                ║
║  • Deploy demand sensing for volatile SKUs (CV > 0.5)            ║
║  • Negotiate VMI (vendor-managed inventory) with top 3 suppliers ║
╚══════════════════════════════════════════════════════════════════╝
"""
print(summary)


╔══════════════════════════════════════════════════════════════════╗
║         🏭 MANUFACTURING INVENTORY OPTIMIZATION — SUMMARY        ║
╠══════════════════════════════════════════════════════════════════╣
║  📊 CURRENT STATE                                                ║
║  • Total SKUs Analyzed   : 40                                    ║
║  • Stockout SKUs         : 0 (IMMEDIATE action required)          ║
║  • Critical SKUs         : 32 (order within lead time)              ║
║  • Overstock SKUs        : 0                                     ║
║  • Total Annual Hold Cost: $   677,239                       ║
╠══════════════════════════════════════════════════════════════════╣
║  🎯 TOP ACTIONS                                                  ║
║  1. 🚨 Emergency Orders: 32 SKUs need immediate replenishment   ║
║  2. 💰 Overstock Reduction: Save ~$0/yr                ║
║  3. 🏭 Audit Supplier: Supplier_B has highest risk score                ║
║  4. 📐 Adopt EOQ model for all A-class SKU

In [46]:
# ── EXPORT ACTION TABLE TO CSV ────────────────────────────
priority_df.to_csv('inventory_action_plan.csv', index=False)
print('✅ Action plan exported to: inventory_action_plan.csv')
print(f'   → {len(priority_df)} SKUs flagged with specific recommended actions')

✅ Action plan exported to: inventory_action_plan.csv
   → 35 SKUs flagged with specific recommended actions
